# Chapter 5: Pretraining on Unlabeled Data

Companion notebook for [Chapter 5 of the interactive course](https://bc-sparknet.github.io/make-an-llm-2/#/chapter/pretraining). The code cells follow the chapter's examples in order; the explanations are on the site.

*This notebook is generated from the chapter source. To change it, edit the chapter and run `npm run notebooks`.*

In [ ]:
%pip install -q torch tiktoken

## Setup

Code carried over from earlier chapters, plus anything this chapter needs to run. Run it once and move on.

In [ ]:
#@title Setup (run me first)

import math
import tiktoken
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- from Chapter 2 ----
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

class NextTokenDataset(Dataset):
    def __init__(self, text, tokenizer, context_length, stride):
        ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
        self.inputs, self.targets = [], []
        for start in range(0, len(ids) - context_length, stride):
            window = ids[start : start + context_length + 1]
            self.inputs.append(torch.tensor(window[:-1]))   # all but the last
            self.targets.append(torch.tensor(window[1:]))   # all but the first

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, i):
        return self.inputs[i], self.targets[i]


def create_dataloader(text, batch_size=8, max_length=256, stride=256,
                      shuffle=True, drop_last=True):
    dataset = NextTokenDataset(text, tiktoken.get_encoding("gpt2"),
                               context_length=max_length, stride=stride)
    return DataLoader(dataset, batch_size=batch_size,
                      shuffle=shuffle, drop_last=drop_last)

# ---- from Chapter 3 ----
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.to_q = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.to_k = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.to_v = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        mask = torch.triu(torch.ones(context_length, context_length, dtype=torch.bool), diagonal=1)
        self.register_buffer("mask", mask)

    def split_heads(self, t):
        b, n, _ = t.shape
        # (b, n, d_out) -> (b, n, heads, head_dim) -> (b, heads, n, head_dim)
        return t.view(b, n, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, x):
        b, n, _ = x.shape
        q = self.split_heads(self.to_q(x))
        k = self.split_heads(self.to_k(x))
        v = self.split_heads(self.to_v(x))

        scores = q @ k.transpose(2, 3)                    # (b, heads, n, n)
        scores = scores.masked_fill(self.mask[:n, :n], float('-inf'))
        weights = torch.softmax(scores / self.head_dim ** 0.5, dim=-1)
        weights = self.dropout(weights)

        out = weights @ v                                 # (b, heads, n, head_dim)
        out = out.transpose(1, 2).contiguous().view(b, n, -1)  # concat heads
        return self.out_proj(out)                         # (b, n, d_out)

# ---- from Chapter 4 ----
GPT_CONFIG_124M = {
    "vocab_size": 50257,     # number of tokens the BPE tokenizer knows
    "context_length": 1024,  # longest input the model can look at
    "emb_dim": 768,          # length of every token vector
    "n_heads": 12,           # attention heads per block
    "n_layers": 12,          # how many transformer blocks to stack
    "drop_rate": 0.1,        # dropout probability
    "qkv_bias": True,        # bias terms in the query/key/value projections
}

class LayerNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))   # γ
        self.shift = nn.Parameter(torch.zeros(dim))  # β

    def forward(self, x):
        mu = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_hat = (x - mu) / torch.sqrt(var + self.eps)
        return self.scale * x_hat + self.shift

import math

class GELU(nn.Module):
    def forward(self, x):
        c = math.sqrt(2.0 / math.pi)
        return 0.5 * x * (1.0 + torch.tanh(c * (x + 0.044715 * x ** 3)))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg["emb_dim"]
        self.net = nn.Sequential(
            nn.Linear(d, 4 * d),  # expand:  768 → 3072
            GELU(),
            nn.Linear(4 * d, d),  # project: 3072 → 768
        )

    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg["emb_dim"]
        self.norm1 = LayerNorm(d)
        self.attn = MultiHeadAttention(
            d_in=d,
            d_out=d,
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.norm2 = LayerNorm(d)
        self.ff = FeedForward(cfg)
        self.drop = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # First half: tokens exchange information (attention) + shortcut
        x = x + self.drop(self.attn(self.norm1(x)))
        # Second half: each token is processed on its own (FFN) + shortcut
        x = x + self.drop(self.ff(self.norm2(x)))
        return x

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop = nn.Dropout(cfg["drop_rate"])
        self.blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, token_ids):
        n_tokens = token_ids.shape[1]
        positions = torch.arange(n_tokens, device=token_ids.device)
        x = self.tok_emb(token_ids) + self.pos_emb(positions)  # [batch, n_tokens, 768]
        x = self.drop(x)
        x = self.blocks(x)                                     # 12 blocks, shape unchanged
        x = self.final_norm(x)
        return self.out_head(x)                                # [batch, n_tokens, 50257]

def generate_greedy(model, token_ids, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        window = token_ids[:, -context_size:]      # 1. crop
        with torch.no_grad():
            logits = model(window)                 # 2. [batch, n_tokens, vocab]
        last = logits[:, -1, :]                    # 3. next-token scores
        probs = torch.softmax(last, dim=-1)        # 4. probabilities...
        next_id = torch.argmax(probs, dim=-1, keepdim=True)  # ...and the winner
        token_ids = torch.cat([token_ids, next_id], dim=1)   # 5. append
    return token_ids

# The short story this chapter trains on.
with open("short_story.txt", "w", encoding="utf-8") as f:
    f.write("The Keeper of Gull Rock\n\nThe keeper climbed the one hundred and twelve steps of the lighthouse every evening at a quarter past six, whether the sun was still up or long gone. He counted them out loud, not because he might forget the number, but because the counting was the only conversation he could be sure of having each day. One, two, three, all the way to the brass door at the top, where the great lens waited for him like an old dog waiting to be fed.\n\nHis name was Elias Marr, and he had kept the light on Gull Rock for nineteen years. Before that he had been a fisherman, and before that a boy who wanted to be a fisherman, and before that, as far as he could remember, nothing at all. The rock was a mile and a half from the mainland, a hump of black stone with a scrap of grass on its back and a white tower standing on it like a candle on a cake. A supply boat came every second Thursday if the weather allowed, and in winter the weather rarely allowed.\n\nHe did not mind the quiet. The sea was never really quiet anyway. It talked all day, in long sentences that broke against the rocks and started again, and at night it talked in its sleep. The gulls argued. The wind tried every window in turn, like a visitor who had lost his key. Elias had learned to listen to all of it the way other men listened to the radio, half attending, waiting for the part that mattered.\n\nThe part that mattered was always the same: a change. A shift in the swell that meant a storm two days out. A new smell on the wind, of rain or smoke or ice. A different rhythm to the waves at the foot of the tower, which might mean the tide had turned early, or that something large had passed beneath the surface. He wrote these changes in the log, in small square letters, beside the time and the state of the lamp. The log was the only book he owned that he had written himself, and he was secretly proud of it.\n\nOn the evening this story begins, the change was a light.\n\nIt was not his light. His light swept the water every ten seconds, a long white arm reaching out and pulling back, reaching out and pulling back. This other light was small and yellow and it did not move. It sat on the water to the northwest, where there should have been nothing but the dark and the reef the sailors called the Teeth.\n\nElias watched it for a long time. He checked the lamp, trimmed the wick, wound the clockwork that turned the lens, and came back to the window. The yellow light was still there. It had not drifted, which meant it was anchored or aground. No fisherman in his right mind would anchor near the Teeth, and no fisherman in his right mind would be out at all with the barometer falling the way it was.\n\nHe wrote in the log: Six forty. Lamp steady. Unknown light northwest, near the Teeth. Stationary.\n\nThen he put on his coat.\n\nThe keeper's boat was a heavy wooden dinghy with a pair of oars and a small outboard motor that worked when it felt like working. Elias kept it in a stone hut at the bottom of the steps, where a narrow slipway ran down into the water. He dragged the boat out, checked the fuel, and stood for a moment looking at the sky. The clouds were low and moving fast. The swell had grown since the afternoon. If he went, he would have perhaps an hour before the sea became truly dangerous, and the Teeth were twenty minutes away in good conditions.\n\nHe went.\n\nThe motor started on the fourth pull. The boat climbed the first wave, hung for a moment at the top, and slid down the other side, and Elias felt the old fisherman's joy wake up in his chest, the joy of being small and moving and alive on something enormous. Behind him the tower flashed, flashed, flashed. Ahead of him the yellow light grew slowly larger.\n\nIt was a sailboat. As he came closer he could see the shape of it, a small white hull tilted at a bad angle, its mast leaning over the black water. The yellow light was a lantern hung from the boom. The boat had struck the outer edge of the reef and was stuck there, rocking with each wave, making a grinding sound that Elias could feel through the soles of his boots before he could hear it.\n\n\"Hello!\" he shouted. \"Hello the boat!\"\n\nA head appeared over the side. It belonged to a girl of perhaps twelve, with a yellow raincoat and a face that was trying very hard not to look frightened.\n\n\"Are you the lighthouse?\" she called.\n\nIt was such a strange question that Elias almost laughed. \"I'm the keeper,\" he said. \"Close enough. Who else is aboard?\"\n\n\"My grandfather. He hit his head when we struck. He's awake but he keeps asking what day it is.\"\n\nElias brought the dinghy around to the sheltered side of the sailboat, where the hull blocked the worst of the waves. He tied on with a quick hitch and climbed aboard. The deck was slippery and steep. In the small cabin an old man sat on a bunk holding a towel to his forehead. The towel was red in places.\n\n\"Evening,\" said the old man politely. \"Is it Tuesday?\"\n\n\"It's Friday,\" said Elias.\n\n\"Ah,\" said the old man, disappointed, as if he had been hoping for Tuesday.\n\nThe girl's name was Nell. Her grandfather was called Tobias, and he had been sailing these waters for fifty years, and he had never once struck the Teeth before tonight. He told Elias this three times while Elias looked at the cut on his head, which was long but not deep, and at his eyes, which were a little slow to follow a moving finger.\n\n\"We need to get off this boat,\" Elias said. \"The tide's rising, and when it's high the waves will break right over this reef. Your boat will not like that. You will like it even less.\"\n\n\"I'm not leaving her,\" said Tobias.\n\n\"Grandpa,\" said Nell.\n\n\"Fifty years,\" said Tobias, and his voice shook a little. \"She's been with me longer than your grandmother was.\"\n\nElias understood. He had felt the same way about his first fishing boat, and he had cried like a child the day he sold it. But understanding did not change the tide.\n\n\"She'll still be here in the morning,\" he said, which was probably a lie. \"And if she isn't, you'll still be here, and that's the more important of the two. Come on. My boat is small but she's dry, and my tower has a stove and a kettle and a great deal of tea.\"\n\nIt took a long time to get the old man into the dinghy. Nell went first and held the boat steady against the hull while Elias lowered her grandfather down one careful step at a time. A wave lifted everything at the wrong moment and for a terrible second Tobias hung between the two boats with his legs over the black water, and then the wave passed and he was sitting in the bottom of the dinghy with his hat still on his head, looking surprised.\n\n\"Well,\" he said. \"That was exciting.\"\n\nElias untied the line and pulled the starter cord. Nothing happened. He pulled it again. Nothing. The motor, which had worked when it felt like working, had decided that tonight it did not feel like working at all.\n\n\"Oars,\" Elias said, mostly to himself.\n\nHe rowed. He had not rowed a distance like this in years, and within ten minutes his shoulders were burning and his hands were raw. The wind was against them now, pushing the dinghy back toward the reef with every gust. Nell sat in the bow holding the lantern she had taken from the sailboat. Tobias sat in the stern and gave advice, most of which was about sailing and therefore useless in a rowing boat.\n\nAhead of them the lighthouse flashed. Every ten seconds the long white arm swept over them, and for a moment Elias could see everything: the girl's pale face, the old man's bandaged head, the white crests of the waves marching toward them in rows. Then the dark came back, and there was only the yellow lantern and the sound of the oars.\n\n\"How do you know which way to go?\" Nell asked.\n\n\"I follow the light,\" Elias said between strokes.\n\n\"But you're the keeper. Who keeps the light while you're out here?\"\n\nIt was a good question, and Elias did not have a good answer. The clockwork would turn the lens for about four hours before it needed winding again. The lamp had oil enough for the night. The light would keep itself, for a while. But it had never kept itself before, not in nineteen years, and he found that he did not like the idea of it up there all alone, flashing out over an empty sea with nobody to watch it.\n\n\"It's patient,\" he said at last. \"It'll wait for us.\"\n\nThey were halfway home when the rain came. It came all at once, like a door opening, and in seconds they were soaked through. The lantern hissed and sputtered and went out. Nell made a small sound in the dark.\n\n\"It's all right,\" Elias said. \"Watch for the tower. Count with me. One, two, three.\"\n\nThey counted together, and on ten the white arm came around again and lit up the rain like a curtain of falling glass. The tower was closer now. Elias could see the dark shape of the rock beneath it and the pale line of foam where the waves broke on the slipway.\n\n\"Again,\" he said. \"One, two, three.\"\n\nThey counted the whole way in. Tobias joined them after a while, in a deep, cracked voice, and got the numbers wrong, and nobody corrected him. Elias rowed on every count, and on every ten the light came around and showed them that they were a little closer than before. It was, he thought later, the longest conversation he had ever had in the lighthouse, and all of it was numbers.\n\nThe dinghy struck the slipway with a thump that rattled Elias's teeth. He jumped out into water up to his knees and hauled the boat up the stone ramp, and Nell jumped out and helped him, and together they got Tobias onto dry rock and into the hut, where he sat on an upturned crate and announced that he would like that tea now, please.\n\nThe climb up the tower was slow. Tobias managed the first fifty steps well enough and then needed to rest on every landing. Elias did not hurry him. He counted the steps out loud as he always did, and after a while Nell counted too.\n\n\"One hundred and twelve,\" she said when they reached the kitchen at the top of the stairs. \"Do you do that every day?\"\n\n\"Every day.\"\n\n\"Doesn't it get boring?\"\n\nElias considered this while he lit the stove. \"Some days,\" he admitted. \"But the number is always the same, even when everything else changes. There's something comforting about that.\"\n\nHe made tea, strong and sweet, and found dry blankets, and cleaned the cut on Tobias's head properly and bandaged it with the first aid kit that had not been opened since the day it arrived. Then he climbed the last short ladder to the lamp room and wound the clockwork and trimmed the wick and wrote in the log.\n\nNine fifteen. Lamp steady. Recovered two persons from sailing vessel aground on the Teeth. One injured, not seriously. Motor failed. Rowed home. Heavy rain.\n\nHe looked at the words for a while. They seemed very small for the size of the evening. He thought about adding something, but he did not know what, so in the end he just underlined the date.\n\nThe storm lasted two days. On the first day Tobias slept most of the time and woke every few hours to ask what day it was, and Nell told him, and he was always disappointed. On the second day he stopped asking, and ate a large breakfast, and told Elias a long and complicated story about a whale he had seen off the coast of Norway in his youth, and Elias knew then that he was going to be fine.\n\nNell explored the lighthouse from top to bottom. She read every page of the log, including the parts from nineteen years ago when Elias's handwriting had been larger and less sure. She asked him questions about everything: how the lens worked, why the light flashed every ten seconds instead of five or fifteen, what the sailors called the other rocks and reefs along the coast, whether he was ever lonely.\n\n\"The lens is made of rings of glass,\" he told her, \"and each ring bends the light a little, so all of it goes out in one direction instead of spilling everywhere. That's why a small flame can be seen twenty miles away. The flash every ten seconds is this light's signature. Every lighthouse on the coast has a different one, so a sailor who sees a light in the dark can count the flashes and know exactly which light it is, and exactly where he is.\"\n\n\"Like a name,\" said Nell.\n\n\"Like a name,\" Elias agreed.\n\n\"And are you lonely?\"\n\nHe thought about it honestly, because she had asked honestly. \"Sometimes,\" he said. \"Less than you'd think. The sea talks a lot. And there's always the light to look after.\"\n\n\"But the light can't talk back.\"\n\n\"No,\" Elias said. \"It can't. But it listens very well.\"\n\nOn the third morning the wind dropped and the sky cleared and the sea lay flat and bright and innocent, as if it had never done anything wrong in its life. From the lamp room Elias could see the Teeth to the northwest, black and wet in the sunlight. The sailboat was still there. It had been pushed higher onto the reef and lay on its side, but its hull was whole, and its mast was still standing.\n\nTobias looked at it through Elias's binoculars for a long time without saying anything. Then he handed the binoculars back and cleared his throat.\n\n\"She's a stubborn old thing,\" he said. \"Like me.\"\n\nThe supply boat came that afternoon, two days early, because the harbour master on the mainland had seen the lighthouse flashing all through the storm and had noticed, as harbour masters do, a small yellow light near the Teeth that had been there one evening and gone the next. The supply boat carried a doctor, who looked at Tobias's head and said that whoever had bandaged it had done a good job, and a mechanic, who looked at the dinghy's motor and said a number of things that Nell was told not to repeat.\n\nBefore she left, Nell climbed the one hundred and twelve steps one last time, counting out loud, and stood in the lamp room with Elias looking out at the water.\n\n\"Will you write about us in the log?\" she asked.\n\n\"I already have.\"\n\n\"What did you write?\"\n\nHe showed her. She read the short, square sentences and frowned.\n\n\"That's all?\"\n\n\"That's what happened.\"\n\n\"That's not all that happened,\" she said firmly. She picked up his pen, and before he could stop her she wrote underneath his entry, in large round letters: The keeper followed the light and brought us home. We counted all the way.\n\nElias looked at the words for a long time after the supply boat had gone. They did not match his handwriting, and they broke every rule he had ever made for the log, and he found that he did not mind at all.\n\nThat evening, at a quarter past six, the keeper climbed the one hundred and twelve steps of the lighthouse. He counted them out loud, as he always did. But somewhere around the fiftieth step he noticed that he was not counting alone in his head anymore. He was counting with a girl in a yellow raincoat and an old man who got the numbers wrong, and the counting felt less like talking to himself and more like talking to someone who would, one day, perhaps, come back and count with him again.\n\nAt the top he wound the clockwork, trimmed the wick, and lit the lamp. The great lens began to turn. The long white arm reached out over the water, and pulled back, and reached out again, patient as ever, saying its name to anyone who might be out there in the dark.\n\nEvery ten seconds. One, two, three.\n\nMonths later a letter arrived with the supply boat, addressed in large round handwriting to The Keeper, Gull Rock Light. Inside was a drawing of the lighthouse, with rays of yellow crayon coming out of the top, and a small white sailboat sitting upright on a blue sea. Underneath the drawing Nell had written that her grandfather had repaired the boat with the help of three friends and a great deal of complaining, and that it floated again, and that he had promised to sail it past Gull Rock in the summer and wave.\n\nElias pinned the drawing to the wall beside the log. It was the first picture he had ever put up in the lighthouse, and for the first few days it looked strange to him, a bright square of colour in a room that had been grey and white for nineteen years. Then he stopped noticing it, the way people stop noticing things that belong.\n\nSummer came slowly that year. The fog lingered in the mornings and the wind kept a cold edge well into June. Elias painted the railings, as he did every summer, and cleaned the salt from the windows of the lamp room, and replaced two cracked panes in the kitchen. He mended the slipway where the winter storms had pulled away a line of stones. He took the dinghy's motor apart, cleaned every piece, and put it back together, and was pleased and slightly suspicious when it started on the first pull.\n\nHe found himself watching the northwest more than usual. He told himself it was because of the Teeth, because a reef that had caught one boat might catch another, and a keeper ought to watch the dangerous places. This was true, but it was not the whole truth, and the log, which had lately begun to contain more than just the state of the lamp, knew it.\n\nOn the last day of July, in the middle of a clear blue afternoon, a small white sailboat came around the headland to the north. It came slowly, its sails full of a soft westerly breeze, and it gave the Teeth a very wide berth indeed.\n\nElias watched it from the lamp room. He did not need the binoculars to know it, but he used them anyway. There was an old man at the tiller with a white hat pulled low over his eyes, and in the bow there was a small figure in a yellow raincoat, even though the sky was perfectly clear and there was not a drop of rain for a hundred miles.\n\nAs the boat drew level with Gull Rock, the figure in the bow stood up and waved both arms over her head. The old man at the tiller raised one hand.\n\nElias did not have anything to wave, so he did the only thing a keeper can do in the middle of a sunny afternoon. He climbed the last ladder, lit the lamp, and set the great lens turning, and for ten long minutes the light on Gull Rock flashed out over the bright blue water, every ten seconds, saying its name in broad daylight to a small white boat that already knew exactly where it was.\n\nIn the log that evening he wrote the time, the state of the lamp, and the weather. Then, underneath, in his own small square letters, he added one more line.\n\nThey came back. We counted.\n")

## Evaluating generative models

In [ ]:
import tiktoken
import torch

tokenizer = tiktoken.get_encoding("gpt2")

def text_to_token_ids(text, tokenizer):
    ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    return torch.tensor(ids).unsqueeze(0)     # add a batch dimension

def token_ids_to_text(token_ids, tokenizer):
    ids = token_ids.squeeze(0).tolist()       # drop the batch dimension
    return tokenizer.decode(ids)

In [ ]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval()

out = generate_greedy(
    model=model,
    token_ids=text_to_token_ids("The keeper climbed", tokenizer),
    max_new_tokens=8,
    context_size=GPT_CONFIG_124M["context_length"],
)
print(token_ids_to_text(out, tokenizer))
# The keeper climbed Mob sparing Tactical Kn...  (random words)

### Inputs and targets

In [ ]:
# (token IDs here are illustrative)
inputs  = torch.tensor([[464, 12986, 19952],     # "The keeper climbed"
                        [40,   1392,   340]])    # "I saw it"

targets = torch.tensor([[12986, 19952, 262],     # "keeper climbed the"
                        [1392,   340,  13]])     # "saw it ."

## Measuring the loss

In [ ]:
with torch.no_grad():
    logits = model(inputs)                    # (batch, seq_len, vocab)

probas = torch.softmax(logits, dim=-1)        # 1. probabilities

# 2. probability of the correct token at every position
target_probas = probas.gather(dim=-1, index=targets.unsqueeze(-1)).squeeze(-1)

log_probas = torch.log(target_probas)         # 3. log
avg_log_proba = log_probas.mean()             # 4. average
loss = -avg_log_proba                         # 5. negate
print(loss)   # ≈ 10.8 for an untrained model

### Why the log?

### The built-in version

In [ ]:
import torch.nn.functional as F

logits_flat  = logits.flatten(0, 1)    # (batch * seq_len, vocab)
targets_flat = targets.flatten()       # (batch * seq_len,)

loss = F.cross_entropy(logits_flat, targets_flat)
print(loss)   # same value as the manual version

### Perplexity

In [ ]:
perplexity = torch.exp(loss)   # ≈ 48,700 for the untrained model

## Training and validation loss

In [ ]:
with open("short_story.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

print(len(text_data), len(tokenizer.encode(text_data)))
# about 18,000 characters and 4,300 tokens

In [ ]:
split = int(0.9 * len(text_data))
train_text, val_text = text_data[:split], text_data[split:]

train_loader = create_dataloader(
    train_text, batch_size=2, max_length=256, stride=256,
    shuffle=True, drop_last=True,
)
val_loader = create_dataloader(
    val_text, batch_size=2, max_length=256, stride=256,
    shuffle=False, drop_last=False,
)

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    return F.cross_entropy(logits.flatten(0, 1), target_batch.flatten())


def calc_loss_loader(data_loader, model, device, num_batches=None):
    if len(data_loader) == 0:
        return float("nan")
    n = len(data_loader) if num_batches is None else min(num_batches, len(data_loader))
    total = 0.0
    for i, (x, y) in enumerate(data_loader):
        if i >= n:
            break
        total += calc_loss_batch(x, y, model, device).item()
    return total / n

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

with torch.no_grad():
    print(calc_loss_loader(train_loader, model, device))   # ≈ 10.99
    print(calc_loss_loader(val_loader, model, device))     # ≈ 10.98

## The training loop

In [ ]:
def train_model_simple(model, train_loader, val_loader, optimizer, device,
                       num_epochs, eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, tokens_seen = [], [], []
    seen, step = 0, -1

    for epoch in range(num_epochs):
        model.train()                              # enable dropout
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()                  # clear old gradients
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()                        # compute new gradients
            optimizer.step()                       # update the weights
            seen += input_batch.numel()
            step += 1

            if step % eval_freq == 0:
                tr, va = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                train_losses.append(tr)
                val_losses.append(va)
                tokens_seen.append(seen)
                print(f"epoch {epoch+1} step {step:05d}: train {tr:.3f}, val {va:.3f}")

        # after every epoch, look at what the model writes
        generate_and_print_sample(model, tokenizer, device, start_context)

    return train_losses, val_losses, tokens_seen

In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss


def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    idx = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        out = generate_greedy(model, idx, max_new_tokens=40, context_size=context_size)
    print(token_ids_to_text(out, tokenizer).replace("\n", " "))
    model.train()

In [ ]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=0.1)

train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=10, eval_freq=5, eval_iter=5,
    start_context="The keeper climbed", tokenizer=tokenizer,
)

### What just happened?

## Decoding strategies

### Temperature

### Top-k sampling

### Putting it together

In [ ]:
def generate(model, idx, max_new_tokens, context_size,
             temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]          # crop to the context window
        with torch.no_grad():
            logits = model(idx_cond)[:, -1, :]      # logits for the last position

        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            cutoff = top_logits[:, -1:]
            logits = torch.where(logits < cutoff,
                                 torch.full_like(logits, float("-inf")), logits)

        if temperature > 0.0:
            probas = torch.softmax(logits / temperature, dim=-1)
            idx_next = torch.multinomial(probas, num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        if eos_id is not None and (idx_next == eos_id).all():
            break                                   # stop at end-of-text
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

In [ ]:
torch.manual_seed(123)
out = generate(
    model=model,
    idx=text_to_token_ids("The keeper climbed", tokenizer).to(device),
    max_new_tokens=20,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=25,
    temperature=1.4,
)
print(token_ids_to_text(out, tokenizer))

## Saving and loading weights

In [ ]:
torch.save(model.state_dict(), "model.pth")

model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(torch.load("model.pth", map_location=device))
model.eval()      # switch off dropout for inference

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
}, "model_and_optimizer.pth")

checkpoint = torch.load("model_and_optimizer.pth", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
model.train()

### Loading pretrained GPT-2 weights

In [ ]:
import json
import os
import urllib.request
from safetensors.torch import load_file

GPT2_REPOS = {"124M": "gpt2", "355M": "gpt2-medium", "774M": "gpt2-large", "1558M": "gpt2-xl"}

def download_and_load_gpt2(model_size, models_dir="gpt2"):
    """Downloads GPT-2 weights once, then returns (settings, params) from disk."""
    folder = os.path.join(models_dir, model_size)
    os.makedirs(folder, exist_ok=True)
    for file in ("config.json", "model.safetensors"):
        path = os.path.join(folder, file)
        if not os.path.exists(path):
            url = f"https://huggingface.co/openai-community/{GPT2_REPOS[model_size]}/resolve/main/{file}"
            print("Downloading", url)
            urllib.request.urlretrieve(url, path)
    with open(os.path.join(folder, "config.json")) as f:
        settings = json.load(f)
    params = load_file(os.path.join(folder, "model.safetensors"))  # name -> tensor
    return settings, params

In [ ]:
def assign(param, tensor):
    if param.shape != tensor.shape:
        raise ValueError(f"Shape mismatch: model {tuple(param.shape)} vs checkpoint {tuple(tensor.shape)}")
    return torch.nn.Parameter(tensor.detach().clone())

def load_weights_into_gpt(gpt, params):
    p = {k.removeprefix("transformer."): v for k, v in params.items()}
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, p["wte.weight"])
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, p["wpe.weight"])

    for i, block in enumerate(gpt.blocks):
        h = f"h.{i}."
        # OpenAI stores linear weights as (in, out); nn.Linear expects (out, in), hence .T
        # Queries, keys and values live in one combined matrix: split it in three.
        q_w, k_w, v_w = p[h + "attn.c_attn.weight"].chunk(3, dim=-1)
        q_b, k_b, v_b = p[h + "attn.c_attn.bias"].chunk(3, dim=-1)
        attn = block.attn
        attn.to_q.weight, attn.to_q.bias = assign(attn.to_q.weight, q_w.T), assign(attn.to_q.bias, q_b)
        attn.to_k.weight, attn.to_k.bias = assign(attn.to_k.weight, k_w.T), assign(attn.to_k.bias, k_b)
        attn.to_v.weight, attn.to_v.bias = assign(attn.to_v.weight, v_w.T), assign(attn.to_v.bias, v_b)
        attn.out_proj.weight = assign(attn.out_proj.weight, p[h + "attn.c_proj.weight"].T)
        attn.out_proj.bias = assign(attn.out_proj.bias, p[h + "attn.c_proj.bias"])

        expand, project = block.ff.net[0], block.ff.net[2]
        expand.weight = assign(expand.weight, p[h + "mlp.c_fc.weight"].T)
        expand.bias = assign(expand.bias, p[h + "mlp.c_fc.bias"])
        project.weight = assign(project.weight, p[h + "mlp.c_proj.weight"].T)
        project.bias = assign(project.bias, p[h + "mlp.c_proj.bias"])

        block.norm1.scale = assign(block.norm1.scale, p[h + "ln_1.weight"])
        block.norm1.shift = assign(block.norm1.shift, p[h + "ln_1.bias"])
        block.norm2.scale = assign(block.norm2.scale, p[h + "ln_2.weight"])
        block.norm2.shift = assign(block.norm2.shift, p[h + "ln_2.bias"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, p["ln_f.weight"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, p["ln_f.bias"])
    gpt.out_head.weight = assign(gpt.out_head.weight, p["wte.weight"])  # GPT-2 ties these

In [ ]:
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

gpt = GPTModel(GPT_CONFIG_124M)
load_weights_into_gpt(gpt, params)
gpt.to(device).eval()

torch.manual_seed(123)
out = generate(
    model=gpt,
    idx=text_to_token_ids("The keeper climbed", tokenizer).to(device),
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=50,
    temperature=1.0,
)
print(token_ids_to_text(out, tokenizer))